# ReAct agent Pattern - LlamaIndex RAG Integration 

In [ ]:
# Install required packages (if not already installed)
%pip install llama-index
%pip install llama-index-agent-openai  # For ReActAgent
%pip install llama-index-llms-huggingface-api
%pip install llama-index-embeddings-huggingface
%pip install llama-index-core
%pip install llama-index-llms-groq
%pip install llama-index-vector-stores-chroma
%pip install chromadb

## Relevant imports and Groq Client

We start by importing all the libraries we'll be using in this tutorial as well as the Groq client.

In [ ]:
import sys
from pathlib import Path

# Get the absolute path to the project root
# Assuming notebook is in 'notebooks/' and src is in 'src/' at the project root
project_root = Path.cwd().parent.resolve()
src_path = project_root / "src"

# Add src to sys.path if it's not already there
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    print(f"✅ Added {src_path} to sys.path")

In [ ]:
import os
import re
import math
import json
from dotenv import load_dotenv

from pathlib import Path
from typing import List, Dict, Any
from IPython.display import display, Markdown
import time

# LlamaIndex core imports
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    Settings,
    load_index_from_storage
)
from llama_index.core.agent import ReActAgent, AgentWorkflow
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

# ChromaDB for vector storage
import chromadb

#from groq import Groq

from agent.tool import tool
from agent.utils.extraction import extract_tag_content


# Remember to load the environment variables. 
load_dotenv()

MODEL = "llama-3.3-70b-versatile"
#GROQ_CLIENT = Groq()

In [ ]:
# Set up paths
PROJECT_ROOT = Path("..")
VECTOR_DB_DIR = PROJECT_ROOT / "data" / "vector_db"
SAMPLE_DATA_DIR = PROJECT_ROOT / "resources" / "sample-datasets"

# Create necessary directories
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"💾 Vector DB Directory: {VECTOR_DB_DIR}")
print(f"📄 Sample Data Directory: {SAMPLE_DATA_DIR}")
print(f"\n✅ Paths configured successfully!")

In [ ]:

# Set up embedding model
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",  # Lightweight, high-quality embedding model
    cache_folder=str(PROJECT_ROOT / "models")
)

# You can specify the model you want to use, e.g., "llama-3.3-70b-versatile"
# If you don't specify a model, it defaults to "mixtral-8x7b-32768"
llm = Groq(model="llama-3.3-70b-versatile")

# Configure global settings
Settings.embed_model = embed_model
Settings.llm = llm
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print("✅ LlamaIndex settings configured:")
print(f"   - Embedding Model: BAAI/bge-small-en-v1.5")
print(f"   - LLM: Groq llama-3.3-70b-versatile")
print(f"   - Chunk Size: 512")
print(f"   - Chunk Overlap: 50")

In [ ]:
# Load existing vector index from Phase 1
print("🔄 Loading existing vector index from Phase 1...")

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))
collection_name = "internal_knowledge_base"

# Load the collection
try:
    chroma_collection = chroma_client.get_collection(name=collection_name)
    print(f"✅ Found existing collection: {collection_name}")
    print(f"   Total vectors: {chroma_collection.count()}")
except Exception as e:
    print(f"❌ Error loading collection: {e}")
    print("   Please run Phase 1 notebook first to create the vector index.")
    raise

# Create ChromaVectorStore wrapper
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create storage context
storage_context = StorageContext.from_defaults(vector_store=vector_store, persist_dir=str(VECTOR_DB_DIR))

# Load the index
try:
    index = load_index_from_storage(storage_context)
    print("✅ Vector index loaded successfully!")
except Exception as e:
    print(f"❌ Error loading index: {e}")
    raise

### Defining the Tools

Let's build an RAG tool that involves the use of a rag_query_engine tool.

In [ ]:
# Create QueryEngineTool from the vector index
print("🔧 Setting up QueryEngine Tool...")

# Create query engine with optimized settings for agent use
query_engine = index.as_query_engine(
    similarity_top_k=5,      # Retrieve top 5 most similar chunks
    response_mode="compact", # Concatenate chunks and generate single response
    streaming=False
)

@tool
def rag_query_engine(query: str) -> str:
    """
    A tool to query the knowledge base containing internal company documents. 

    Args:
        query (str): The query string to search the knowledge base.

    Returns:
        str: The response from the knowledge base along with citation sources.
        example: According to our HR policies, parental leave is 16 weeks. [Sources: company_handbook.md].
    """

    citations = []
    response = query_engine.query(query)

    if response and hasattr(response, "source_nodes"):
        citations = [node.node.metadata.get('file_name', 'Unknown') for node in response.source_nodes]

    final_answer = str(response) + f"\n\n[Sources: {', '.join(citations)}" + "]"
    return final_answer


available_tools = {
    "rag_query_engine": rag_query_engine
}

The `@tool` operator allows us to convert a Python function into a `Tool` automatically

In [ ]:
print("Tool name: ", rag_query_engine.name)
print("Tool signature: ", rag_query_engine.fn_signature)

## Using the `ReactAgent` library 

In [ ]:
from agent.react_agent import ReactAgent

In [ ]:
agent = ReactAgent(tools=[rag_query_engine])

In [ ]:
response = agent.run(user_msg='How do I set up the local dev environment for project Nexus?')

In [ ]:
# Display the agent's final response to the user
print(response)

---

ReAct Agent - LlamaIndex Integration working as expected! 🚀🚀🚀🚀